In [3]:
%pip install pandas mysql-connector-python pyarrow streamlit matlotlib seaborn

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement matlotlib (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for matlotlib


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv(r'C:\Users\camil\OneDrive\Escritorio\Proyecto_final_mod2\olist_order_items_dataset.csv')
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nBasic statistics:")
print(df.describe())

In [ ]:
pd.to_datetime(df['shipping_limit_date'], errors='coerce')

In [ ]:
# Create Father Table: customers
father_table = df[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']].drop_duplicates()

# Create Daughter Table: customer_addresses
daughter_table = df[['customer_id', 'customer_zip_code_prefix']].drop_duplicates()

# Save to CSV files
father_table.to_csv("customers.csv", index=False)
daughter_table.to_csv("customer_addresses.csv", index=False)

print("Father table (customers):")
print(father_table.head())
print("\nDaughter table (customer_addresses):")
print(daughter_table.head())

In [ ]:
CREATE TABLE customers (
    customer_id   VARCHAR(50) PRIMARY KEY,
    customer_city VARCHAR(100),
    customer_state VARCHAR(5)
);

CREATE TABLE orders (
    order_id      VARCHAR(50) PRIMARY KEY,
    customer_id   VARCHAR(50),
    order_status  VARCHAR(20),
    order_purchase_timestamp    DATETIME,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE order_items (
    order_id      VARCHAR(50) PRIMARY KEY,
    order_item_id VARCHAR(50),
    product_id    VARCHAR(50),
    seller_id     VARCHAR(50),
    shipping_limit_date DATETIME,
    price         DECIMAL(10,2),
    freight_value DECIMAL(10,2),
    FOREIGN KEY (order_id) REFERENCES orders(order_id)
);

### Ejemplo de uso (adaptar columnas a vuestro CSV):
### df_customers = pd.read_csv("olist_customers_dataset.csv")
### df_customers = df_customers[["customer_id", "customer_city", "customer_state"]]
### conn = mysql.connector.connect(**mysql_config)
### cargar_csv_en_tabla(df_customers, "customers", conn)
### conn.close()

In [ ]:
# Add this before the data processing section:
df_order_items = df_order_items.drop_duplicates(subset=['order_id', 'order_item_id'], keep='first')

In [ ]:
import pandas as pd
import mysql.connector
from credenciales import mysql_config

def crear_tabla_orders(conn):
    """Crea la tabla orders si no existe"""
    cursor = conn.cursor()
    create_table_query = """
    CREATE TABLE IF NOT EXISTS orders (
        order_id VARCHAR(50) PRIMARY KEY,
        customer_id VARCHAR(50),
        order_status VARCHAR(20),
        order_purchase_timestamp DATETIME,
        order_delivered_carrier_date DATETIME,
        order_delivered_customer_date DATETIME,
        order_estimated_delivery_date DATETIME
    )
    """
    cursor.execute(create_table_query)
    conn.commit()
    cursor.close()
    print("Tabla orders creada o ya existente")

def crear_tabla_order_items(conn):
    """Crea la tabla order_items si no existe"""
    cursor = conn.cursor()
    create_table_query = """
    CREATE TABLE IF NOT EXISTS order_items (
        order_id VARCHAR(50),
        order_item_id VARCHAR(50),
        product_id VARCHAR(50),
        seller_id VARCHAR(50),
        shipping_limit_date DATETIME,
        price DECIMAL(10,2),
        freight_value DECIMAL(10,2),
        PRIMARY KEY (order_id, order_item_id),
        FOREIGN KEY (order_id) REFERENCES orders(order_id)
    )
    """
    cursor.execute(create_table_query)
    conn.commit()
    cursor.close()
    print("Tabla order_items creada o ya existente")

def cargar_olist_order_items_dataset_en_order_items(df, order_items, conn, batch=1000):
    """Inserta un DataFrame en una tabla de MySQL en lotes de `batch` filas."""
    cursor = conn.cursor()
    cols = ", ".join(df.columns)
    placeholders = ", ".join(["%s"] * len(df.columns))
    # Use INSERT IGNORE to skip duplicates
    sql = f"INSERT IGNORE INTO {order_items} ({cols}) VALUES ({placeholders})"

    filas = [tuple(None if pd.isna(v) else v for v in row)
             for row in df.itertuples(index=False, name=None)]

    for i in range(0, len(filas), batch):
        cursor.executemany(sql, filas[i:i+batch])
        conn.commit()
        print(f"  insertadas {min(i+batch, len(filas))}/{len(filas)}")
    cursor.close()

try:
    # Load and process the data
    df_order_items = pd.read_csv(r"C:\Users\camil\OneDrive\Escritorio\Proyecto_final_mod2\olist_order_items_dataset.csv")
    df_order_items = df_order_items[["order_id", "order_item_id", "product_id", "seller_id", "shipping_limit_date", "price", "freight_value"]]

    # Connect to database
    conn = mysql.connector.connect(**mysql_config)

    # Create tables first (orders before order_items)
    crear_tabla_orders(conn)
    crear_tabla_order_items(conn)

    # Call the function to insert data
    cargar_olist_order_items_dataset_en_order_items(df_order_items, "order_items", conn)

    print("Datos cargados exitosamente!")
    
except mysql.connector.Error as err:
    print(f"Error de MySQL: {err}")
except Exception as e:
    print(f"Error: {e}")
finally:
    if 'conn' in locals() and conn.is_connected():
        conn.close()
        print("Conexión cerrada")


In [ ]:
conn = mysql.connector.connect(**mysql_config)

query = """
SELECT
    o.order_id,
    o.order_purchase_timestamp as order_date,  -- Fixed: changed from order_date to order_purchase_timestamp
    o.order_status,
    c.customer_city,
    c.customer_state,
    oi.price,
    oi.freight_value
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
"""

df = pd.read_sql(query, conn)
conn.close()

df.head()


In [ ]:
df.to_pickle("dataset_analitico.pkl")
df = pd.read_pickle("dataset_analitico.pkl")

# Fase 5

In [ ]:
query = """
SELECT
    o.order_id,
    o.order_purchase_timestamp as order_date,  -- Fixed: changed from order_date to order_purchase_timestamp
    o.order_status,
    c.customer_city,
    c.customer_state,
    oi.price,
    oi.freight_value
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
"""
df.head(5)

# Fase 6

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe()

In [ ]:
df.head(20)


# Fase 7

In [ ]:
# Here we will analyze the total number of orders by state. We will group the data by 'customer_state' and count the number of orders for each state.
orders_by_state = df.groupby('customer_state').size().reset_index(name='total_orders')
orders_by_state = orders_by_state.sort_values('total_orders', ascending=False)

print("Total number of orders by state:")
print(orders_by_state)



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better looking plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Create the bar chart
plt.figure(figsize=(12, 6))
bars = plt.bar(orders_by_state['customer_state'], orders_by_state['total_orders'])

# Customize the plot
plt.title('Total Number of Orders by State', fontsize=16, fontweight='bold')
plt.xlabel('State', fontsize=12)
plt.ylabel('Number of Orders', fontsize=12)
plt.xticks(rotation=45, ha='right')

# Add value labels on bars (optional)
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}',
             ha='center', va='bottom')

plt.tight_layout()
plt.show()


Aquí podemos ver que estados tuvieron mayor número de ordenes totales ordenados de mayor a menor

In [ ]:
# Calculate total sales by state (sum of price column)
sales_by_state = df.groupby('customer_state')['price'].sum().reset_index()
sales_by_state = sales_by_state.sort_values('price', ascending=False)

print("Total sales by state (sorted by highest sales):")
print(sales_by_state)

# Find the state with highest sales
max_sales_state = sales_by_state.loc[sales_by_state['price'].idxmax()]
print(f"\nState with highest sales: {max_sales_state['customer_state']}")
print(f"Total sales: R${max_sales_state['price']:,.2f}")

# Create a bar chart for sales by state
plt.figure(figsize=(12, 6))
bars = plt.bar(sales_by_state['customer_state'], sales_by_state['price'])
plt.title('Total Sales by State', fontsize=16, fontweight='bold')
plt.xlabel('State', fontsize=12)
plt.ylabel('Total Sales (R$)', fontsize=12)
plt.xticks(rotation=45, ha='right')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'R${height:,.0f}',
             ha='center', va='bottom')

plt.tight_layout()
plt.show()


Y aquí podemos ver los estados que mas vendieron en dinero, igualmente ordenados de mayor a menor

Como podemos ver, hay una clara dominancia por parte de SP sobre los demás estados

In [ ]:
# Group by date and count orders for SP state
sp_orders_by_date = df[df['customer_state'] == 'SP'].groupby('order_date').size().reset_index(name='order_count')

# Sort by date ascending (oldest to newest)
sp_orders_by_date = sp_orders_by_date.sort_values('order_date')

# Create line chart
plt.figure(figsize=(15, 6))
plt.plot(sp_orders_by_date['order_date'], sp_orders_by_date['order_count'], marker='o', linewidth=2, markersize=4, color='red')
plt.title('Number of Orders by Date for SP State', fontsize=14, fontweight='bold')
plt.xlabel('Order Date')
plt.ylabel('Number of Orders')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Display the results
print("Total number of orders for SP over time:")
print(sp_orders_by_date.to_string(index=False))

# Show total count
print(f"\nTotal orders for SP: {sp_orders_by_date['order_count'].sum()}")


In [ ]:
# Group by date and sum price for SP state
sp_sales_by_date = df[df['customer_state'] == 'SP'].groupby('order_date')['price'].sum().reset_index()

# Sort by date ascending (oldest to newest)
sp_sales_by_date = sp_sales_by_date.sort_values('order_date')

# Create line chart
plt.figure(figsize=(15, 6))
plt.plot(sp_sales_by_date['order_date'], sp_sales_by_date['price'], marker='o', linewidth=2, markersize=4, color='blue')
plt.title('Total Sales by Date for SP State', fontsize=14, fontweight='bold')
plt.xlabel('Order Date')
plt.ylabel('Total Sales (R$)')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Display the results
print("Total sales for SP over time:")
print(sp_sales_by_date.to_string(index=False))

# Show total sales
print(f"\nTotal sales for SP: R${sp_sales_by_date['price'].sum():,.2f}")



Usando estas dos tablas podemos ver que el numero de pedidos no es directamente proporcional al total de ventas por fecha

In [ ]:
# Get total number of orders per city for SP state
orders_by_city = df[df['customer_state'] == 'SP'].groupby('customer_city').size().reset_index(name='order_count')

# Sort by order count descending and get top 20
orders_by_city = orders_by_city.sort_values('order_count', ascending=False).head(20)

# Create bar chart
plt.figure(figsize=(12, 10))
bars = plt.barh(orders_by_city['customer_city'], orders_by_city['order_count'], color='skyblue')
plt.title('Top 20 Cities by Number of Orders for SP State', fontsize=16, fontweight='bold')
plt.xlabel('Number of Orders', fontsize=12)
plt.ylabel('City', fontsize=12)
plt.grid(True, alpha=0.3, axis='x')

# Add value labels on bars
for i, (city, count) in enumerate(zip(orders_by_city['customer_city'], orders_by_city['order_count'])):
    plt.text(count + 1, i, str(count), va='center')

plt.tight_layout()
plt.show()

# Display the results
print("Top 20 cities by number of orders for SP state:")
print(orders_by_city.to_string(index=False))

# Total orders for SP state
print(f"\nTotal orders for SP state: {orders_by_city['order_count'].sum()}")




In [ ]:
# Get total number of orders per city for all states
orders_by_city = df.groupby(['customer_state', 'customer_city']).size().reset_index(name='order_count')

# Sort by order count descending and get top 20
orders_by_city = orders_by_city.sort_values('order_count', ascending=False).head(20)

# Create bar chart
plt.figure(figsize=(12, 10))
bars = plt.barh(orders_by_city['customer_city'], orders_by_city['order_count'], color='lightgreen')
plt.title('Top 20 Cities by Number of Orders (All States)', fontsize=16, fontweight='bold')
plt.xlabel('Number of Orders', fontsize=12)
plt.ylabel('City', fontsize=12)
plt.grid(True, alpha=0.3, axis='x')

# Add value labels on bars
for i, (city, count) in enumerate(zip(orders_by_city['customer_city'], orders_by_city['order_count'])):
    plt.text(count + 1, i, str(count), va='center')

plt.tight_layout()
plt.show()

# Display the results
print("Top 20 cities by number of orders (All States):")
print(orders_by_city.to_string(index=False))

# Total orders for all states
print(f"\nTotal orders for all states: {orders_by_city['order_count'].sum()}")



In [ ]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# Set page configuration
st.set_page_config(page_title="Brazilian E-commerce Analysis", layout="wide")

# Title
st.title("📊 Brazilian E-commerce Analysis Dashboard")

# Load data function
@st.cache_data
def load_data():
    file_path = r'C:\Users\camil\OneDrive\Escritorio\Proyecto_final_mod2\dataset_analitico.pkl'
    
    try:
        # Check if file exists
        if not os.path.exists(file_path):
            st.error(f"❌ File not found: {file_path}")
            st.info("Creating sample data for demonstration...")
            
            # Create sample data
            np.random.seed(42)
            states = ['SP', 'RJ', 'MG', 'RS', 'PR', 'SC', 'BA', 'CE', 'PE', 'GO']
            cities = ['São Paulo', 'Rio de Janeiro', 'Belo Horizonte', 'Porto Alegre', 'Curitiba', 
                     'Florianópolis', 'Salvador', 'Fortaleza', 'Recife', 'Goiânia']
            
            data = []
            for _ in range(1000):
                state = np.random.choice(states)
                city = np.random.choice(cities)
                data.append({'customer_state': state, 'customer_city': city})
            
            df = pd.DataFrame(data)
            st.success("✅ Sample data created successfully!")
            return df
        
        # Load the pickle file
        df = pd.read_pickle(file_path)
        st.success("✅ Data loaded successfully from file!")
        return df
        
    except Exception as e:
        st.error(f"❌ Error loading data: {str(e)}")
        st.info("Creating sample data for demonstration...")
        
        # Create sample data as fallback
        np.random.seed(42)
        states = ['SP', 'RJ', 'MG', 'RS', 'PR', 'SC', 'BA', 'CE', 'PE', 'GO']
        cities = ['São Paulo', 'Rio de Janeiro', 'Belo Horizonte', 'Porto Alegre', 'Curitiba', 
                 'Florianópolis', 'Salvador', 'Fortaleza', 'Recife', 'Goiânia']
        
        data = []
        for _ in range(1000):
            state = np.random.choice(states)
            city = np.random.choice(cities)
            data.append({'customer_state': state, 'customer_city': city})
        
        df = pd.DataFrame(data)
        st.success("✅ Sample data created successfully!")
        return df

# Load the data
df = load_data()

# Check if df is loaded properly
if df is None or df.empty:
    st.error("❌ No data available to display. Please check the file path or data source.")
    st.stop()

# Sidebar for filters
st.sidebar.header("Filters")

# Ensure the required columns exist
if 'customer_state' not in df.columns:
    st.error("❌ Required column 'customer_state' not found in data")
    st.stop()

if 'customer_city' not in df.columns:
    st.error("❌ Required column 'customer_city' not found in data")
    st.stop()

# Create sidebar filter
selected_state = st.sidebar.selectbox(
    "Select State", 
    ['All'] + list(df['customer_state'].unique())
)

# Filter data based on selection
if selected_state == 'All':
    filtered_df = df
else:
    filtered_df = df[df['customer_state'] == selected_state]

# Main content
st.header(f"Analysis for {selected_state if selected_state != 'All' else 'All States'}")

# Top 20 cities by order count
orders_by_city = filtered_df.groupby(['customer_state', 'customer_city']).size().reset_index(name='order_count')
orders_by_city = orders_by_city.sort_values('order_count', ascending=False).head(20)

# Create bar chart
fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(orders_by_city['customer_city'], orders_by_city['order_count'], color='skyblue')
ax.set_xlabel('Number of Orders')
ax.set_ylabel('City')
ax.set_title(f'Top 20 Cities by Order Count ({selected_state if selected_state != "All" else "All States"})')
ax.invert_yaxis()  # Show highest at top

# Add value labels on bars
for i, (bar, count) in enumerate(zip(bars, orders_by_city['order_count'])):
    ax.text(count + 0.5, i, str(count), va='center')

plt.tight_layout()
st.pyplot(fig)

# Display table
st.subheader("Top 20 Cities by Order Count")
st.dataframe(orders_by_city.reset_index(drop=True))

# Total orders metric
total_orders = orders_by_city['order_count'].sum()
st.metric("Total Orders", f"{total_orders:,}")

# State-wise distribution
st.subheader("Orders by State")
state_orders = df.groupby('customer_state').size().sort_values(ascending=False)
st.bar_chart(state_orders)

# Data information
st.sidebar.markdown("---")
st.sidebar.subheader("Data Info")
st.sidebar.write(f"Total Records: {len(df):,}")
st.sidebar.write(f"Total States: {df['customer_state'].nunique()}")
st.sidebar.write(f"Total Cities: {df['customer_city'].nunique()}")
